# Imports

In [82]:
import requests
from bs4 import BeautifulSoup
import pandas
import unicodedata

# Constants

In [ ]:
base_url = 'https://mibici.net'

# Fetch website

In [ ]:
source_url = f'{base_url}/es/datos-abiertos/'

headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'
}

In [13]:
r = requests.get(source_url, headers=headers)

r

<Response [200]>

In [14]:
r.text

'<!DOCTYPE html><html lang="es"><head><meta charset="utf-8"><meta http-equiv="X-UA-Compatible" content="IE=edge"><meta name="viewport" content="width=device-width,initial-scale=1"><meta name="robots" content="INDEX,FOLLOW"/><title>Mi Bici | Datos abiertos</title><meta name="description" content="Explora, experimenta y comparte los datos de uso de Mi Bici."><link rel="canonical" href="https://mibici.net/es/datos-abiertos/"><meta property="og:title" content="MiBici |Datos abiertos"><meta property="og:description" content="Explora, experimenta y comparte los datos de uso de Mi Bici."><meta property="og:image" content="https://www.mibici.net/site/templates/assets/images/og-meta-image.jpg"><meta property="og:image:type" content="image/jpeg"><meta property="og:image:width" content="960"><meta property="og:image:height" content="540"><meta property="og:type" content="website"><meta property="og:site_name" content="Mi Bici"><meta property="og:url" content="https://mibici.net/es/datos-abiertos/

# Parse HTML for file links

In [16]:
soup = BeautifulSoup(r.text, 'html.parser')

In [ ]:
links = soup.find_all('a')

In [34]:
parent_dir = '/site/assets/files'

file_links = []
file_paths = []

for link in links:
    link_href = link.get('href')
    if not link_href.startswith(parent_dir): continue
    if not 'datos_abiertos' in link_href: continue
    file_links.append(link)
    file_paths.append(link_href)

# file_links = [link for link in links if link.get('href').startswith(parent_dir) and ('datos_abiertos' in link.get('href'))]
# links[10].get('href')

# Filter by date range

In [62]:
df_paths = pandas.DataFrame(file_paths, columns=['path'])

df_paths.head()

,path
0,/site/assets/files/1079/datos_abiertos_2014_12...
1,/site/assets/files/1084/datos_abiertos_2015_01...
2,/site/assets/files/1097/datos_abiertos_2015_02...
3,/site/assets/files/1098/datos_abiertos_2015_03...
4,/site/assets/files/1099/datos_abiertos_2015_04...


In [63]:
df_date = ( df_paths.path
    .str.extract(r'(\d{4}_\d{2})')
    .rename(columns={0:'file_date'})
)

df_date.head()

,file_date
0,2014_12
1,2015_01
2,2015_02
3,2015_03
4,2015_04


In [64]:
df_paths['file_date'] = pandas.to_datetime(df_date['file_date'], format='%Y_%m')

df_paths.head()

,path,file_date
0,/site/assets/files/1079/datos_abiertos_2014_12...,2014-12-01
1,/site/assets/files/1084/datos_abiertos_2015_01...,2015-01-01
2,/site/assets/files/1097/datos_abiertos_2015_02...,2015-02-01
3,/site/assets/files/1098/datos_abiertos_2015_03...,2015-03-01
4,/site/assets/files/1099/datos_abiertos_2015_04...,2015-04-01


In [66]:
date_start = '2016-04-01'
date_end = '2016-10-01'
df_paths.query('(file_date >= @date_start) and (file_date <= @date_end)')

,path,file_date
16,/site/assets/files/1029/datos_abiertos_2016_04...,2016-04-01
17,/site/assets/files/1058/datos_abiertos_2016_05...,2016-05-01
18,/site/assets/files/1080/datos_abiertos_2016_06...,2016-06-01
19,/site/assets/files/1081/datos_abiertos_2016_07...,2016-07-01
20,/site/assets/files/1082/datos_abiertos_2016_08...,2016-08-01
21,/site/assets/files/1083/datos_abiertos_2016_09...,2016-09-01
22,/site/assets/files/1108/datos_abiertos_2016_10...,2016-10-01


In [80]:
df_paths.path.to_list()

['/site/assets/files/1079/datos_abiertos_2014_12.csv',
 '/site/assets/files/1084/datos_abiertos_2015_01.csv',
 '/site/assets/files/1097/datos_abiertos_2015_02.csv',
 '/site/assets/files/1098/datos_abiertos_2015_03.csv',
 '/site/assets/files/1099/datos_abiertos_2015_04.csv',
 '/site/assets/files/1100/datos_abiertos_2015_05.csv',
 '/site/assets/files/1101/datos_abiertos_2015_06.csv',
 '/site/assets/files/1102/datos_abiertos_2015_07.csv',
 '/site/assets/files/1103/datos_abiertos_2015_08.csv',
 '/site/assets/files/1104/datos_abiertos_2015_09.csv',
 '/site/assets/files/1105/datos_abiertos_2015_10.csv',
 '/site/assets/files/1106/datos_abiertos_2015_11.csv',
 '/site/assets/files/1107/datos_abiertos_2015_12.csv',
 '/site/assets/files/1020/datos_abiertos_2016_01-1.csv',
 '/site/assets/files/1021/datos_abiertos_2016_02.csv',
 '/site/assets/files/1023/datos_abiertos_2016_03.csv',
 '/site/assets/files/1029/datos_abiertos_2016_04.csv',
 '/site/assets/files/1058/datos_abiertos_2016_05.csv',
 '/site/

# Download a file

In [68]:
filepath = df_paths.loc[0, 'path']

file_url = base_url + filepath

file_url

'https://mibici.net/site/assets/files/1079/datos_abiertos_2014_12.csv'

In [69]:
r_file = requests.get(file_url, headers=headers)

r_file

<Response [200]>

In [76]:
type(r_file.content)

bytes

In [77]:
filename = filepath.split('/')[-1]
print(filename)

with open(filename, 'wb') as f:
    f.write(r_file.content)

datos_abiertos_2014_12.csv


# Read file

In [81]:
df = pandas.read_csv(filename)

df

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,4601,10324,M,NaN,2014-12-01 00:33:47,2014-12-01 00:36:54,47,47
1,4604,10324,M,NaN,2014-12-01 01:06:54,2014-12-01 01:08:45,5,5
2,4628,3,F,1990.0,2014-12-01 09:47:20,2014-12-01 09:47:47,79,79
3,4631,102,M,1982.0,2014-12-01 09:48:23,2014-12-01 09:48:37,79,79
4,4632,3,F,1990.0,2014-12-01 09:48:46,2014-12-01 09:48:57,79,79
...,...,...,...,...,...,...,...,...
23962,42029,1736,M,1966.0,2014-12-31 22:43:35,2014-12-31 22:56:17,51,38
23963,42030,20342,M,NaN,2014-12-31 22:49:43,2014-12-31 23:04:32,77,50
23964,42031,20342,M,NaN,2014-12-31 23:09:37,2014-12-31 23:25:23,50,37
23965,42032,1736,M,1966.0,2014-12-31 23:33:28,2014-12-31 23:39:59,51,52


In [83]:
cols = list(df.columns)
cols

['Viaje_Id',
 'Usuario_Id',
 'Genero',
 'Año_de_nacimiento',
 'Inicio_del_viaje',
 'Fin_del_viaje',
 'Origen_Id',
 'Destino_Id']

In [87]:
for col in cols:
    col_lower = col.lower()
    col_norm = unicodedata.normalize('NFKD', col_lower)
    col_final = col_norm.encode('ascii', 'ignore').decode('ascii')

    print(col, '\n', col_norm, '\n', col_final)
    print('')

Viaje_Id 
 viaje_id 
 viaje_id

Usuario_Id 
 usuario_id 
 usuario_id

Genero 
 genero 
 genero

Año_de_nacimiento 
 año_de_nacimiento 
 ano_de_nacimiento

Inicio_del_viaje 
 inicio_del_viaje 
 inicio_del_viaje

Fin_del_viaje 
 fin_del_viaje 
 fin_del_viaje

Origen_Id 
 origen_id 
 origen_id

Destino_Id 
 destino_id 
 destino_id

